# Part 3: Reverse Mode Automatic Differentiation

Dynamic Reverse mode AD can be implemented by declaring a class to represent a value and the child expressions that the value depends on. We've provided the implementation that was shown in the lecture slides as a basis below, but it's missing some parts that will make it useful.

__Tasks:__

- Addition (`__add__`) is incomplete - can you finish it?
- Can you also implement division (`__truediv__`), subtraction (`__sub__`) and power (`__pow__`)?

In [2]:
import math

class Var:
    def __init__(self, value):
        self.value = value
        self.children = []
        self.grad_value = None

    def grad(self):
        if self.grad_value is None:
            self.grad_value = sum(weight * var.grad()
                                  for weight, var in self.children)
        return self.grad_value

    def __str__(self):
        return str(self.value)

    def __mul__(self, other):
        z = Var(self.value * other.value)
        self.children.append((other.value, z))
        other.children.append((self.value, z))
        return z

    def __add__(self, other):
        z = Var(self.value + other.value)
        self.children.append((1.0, z))
        other.children.append((1.0, z))
        return z

    def __sub__(self, other):
        z = Var(self.value - other.value)
        self.children.append((1.0, z))
        other.children.append((-1.0, z))
        return z

    def __truediv__(self, other):
        z = Var(self.value / other.value)
        self.children.append((1.0 / other.value, z))
        other.children.append((-self.value / other.value ** 2, z))
        return z

    def __pow__(self, other):
        z = Var(self.value ** other.value)
        self.children.append((other.value * self.value ** (other.value - 1), z))
        # d(u^v)/dv = u^v * ln(u); only defined for u > 0
        log_term = z.value * math.log(self.value) if self.value > 0 else 0.0
        other.children.append((log_term, z))
        return z

In [3]:
# Tests

Var(1) + Var(1) / Var(1) - Var(1)**Var(1)


## Implementing math functions

Just like when we were looking at Forward Mode AD, we also need to implement some core math functions. Here's the sine function for a `Var`:

In [4]:
def sin(x):
    z = Var(math.sin(x.value))
    x.children.append((math.cos(x.value), z))
    return z

__Task:__ can you implement the _cosine_ (`cos`), _tangent_ (`tan`), and _exponential_ (`exp`) functions in the code block below?

In [7]:
# TODO: implement additional math functions on Var instances

def cos(x):
    z = Var(math.cos(x.value))
    x.children.append((math.sin(x.value), z))
    return z
    raise NotImplementedError()

def tan(x):
    z = Var(math.tan(x.value))
    x.children.append((math.cos(x.value), z))
    return z
    raise NotImplementedError()

def exp(x):
    z = Var(math.exp(x.value))
    x.children.append((math.exp(x.value), z))
    return z
    raise NotImplementedError()

In [8]:
# Tests
assert cos(Var(0)).value == 1
assert tan(Var(0)).value == 0
assert exp(Var(0)).value == 1


## Time to try it out

We're now in a position to try our implementation.

__Tasks:__

- Try running the following code to compute the value of the function $z=x\cdot y+sin(x)$ given $x=0.5$ and $y=4.2$, together with the derivative $\partial z/\partial x$ at that point.
- Verify that the result is correct by hand-differentiating the function.

In [9]:
x = Var(0.5)
y = Var(4.2)
z = x * y + sin(x)
print('z:', z)

z.grad_value = 1.0 #Note that we have to 'seed' the gradient of z to 1 (e.g. ∂z/∂z=1) before computing grads
print('∂z/∂x:',x.grad())

z: 2.579425538604203
∂z/∂x: 5.077582561890373


__Task:__ Now use the code block below to compute the derivative $\partial z/\partial y$ of the above expression (at the same point $x=0.5, y=4.2$ as above). Store the resultant gradient in the variable `dzdy`. Verify by hand that the result is correct.

In [11]:
x = Var(0.5)
y = Var(4.2)

z = x * y + sin(x)
print('z:', z)

z.grad_value = 1.0
dzdy = y.grad()

print('∂z/∂y:', dzdy)

z: 2.579425538604203
∂z/∂y: 0.5


In [12]:
assert dzdy


## Differentiating Algorithms

Now, let's look at doing something wacky: differentiate an algorithm. For this example, we'll use an algorithm that is in a sense static (in this particular case the upper limit of the for loop is predetermined). However, it is not difficult to see that AD is much more general, and could even be applied to stochastic algorithms (say if we replaced the upper limit of the loop below with `Math.floor(Math.random() * 10)` for example).

__Task:__ Consider the following algorithm and in the box below it manually compute the value of $z$ and the gradient $\partial z/\partial x$ at the end of execution.

In [13]:
x = Var(0.5)
z = Var(1)
for i in range(0,2):
    z = (z + Var(i)) * x * x

Unrolling the loop:
- $i=0$: $z = (1 + 0)\,x^2 = x^2$
- $i=1$: $z = (x^2 + 1)\,x^2 = x^4 + x^2$

At $x = 0.5$:

$z = 0.5^4 + 0.5^2 = 0.0625 + 0.25 = 0.3125$

$\frac{\partial z}{\partial x} = 4x^3 + 2x = 4(0.125) + 2(0.5) = 1.5$

__Task__: Now use the code block below to print out the gradient computed by our reverse AD by storing the result in a variable called `grad`. Does it match?

In [20]:
x = Var(0.5)
z = Var(1)
for i in range(0, 2):
    z = (z + Var(i)) * x * x

z.grad_value = 1.0
grad = x.grad()
print(grad)   # 1.5

1.5


In [21]:
# Tests
assert grad


__Task:__ Finally, use the code block below to experiment and test the other math functions and methods you created.

In [23]:
def check(name, f, df, a):
    """Reverse-mode gradient vs analytic derivative vs central finite difference."""
    x = Var(a)
    z = f(x)
    z.grad_value = 1.0
    ad = x.grad()

    h = 1e-6
    numeric = (f(Var(a + h)).value - f(Var(a - h)).value) / (2 * h)
    print(f"{name:<22} value={z.value:.6f}  AD={ad:.6f}  "
          f"analytic={df(a):.6f}  numeric={numeric:.6f}")

a = 0.7

check("x + 3",        lambda x: x + Var(3.0),   lambda x: 1,            a)
check("x - 3",        lambda x: x - Var(3.0),   lambda x: 1,            a)
check("3 - x",        lambda x: Var(3.0) - x,   lambda x: -1,           a)
check("x * x",        lambda x: x * x,          lambda x: 2 * x,        a)
check("3 / x",        lambda x: Var(3.0) / x,   lambda x: -3 / x**2,    a)
check("x ** 3",       lambda x: x ** Var(3.0),  lambda x: 3 * x**2,     a)
check("2 ** x",       lambda x: Var(2.0) ** x,  lambda x: 2**x * math.log(2), a)
check("x ** x",       lambda x: x ** x,         lambda x: x**x * (math.log(x) + 1), a)
check("sin(x)",       sin,                      lambda x: math.cos(x),  a)
check("x*x / (x+1)",  lambda x: x * x / (x + Var(1.0)),
                      lambda x: (x**2 + 2*x) / (x + 1)**2,             a)

check("cos(x)",       cos,                      lambda x: -math.sin(x),        a)
check("tan(x)",       tan,                      lambda x: 1 / math.cos(x)**2,  a)
check("exp(x)",       exp,                      lambda x: math.exp(x),         a)
check("exp(sin(x))",  lambda x: exp(sin(x)),
                      lambda x: math.cos(x) * math.exp(math.sin(x)),          a)

x + 3                  value=3.700000  AD=1.000000  analytic=1.000000  numeric=1.000000
x - 3                  value=-2.300000  AD=1.000000  analytic=1.000000  numeric=1.000000
3 - x                  value=2.300000  AD=-1.000000  analytic=-1.000000  numeric=-1.000000
x * x                  value=0.490000  AD=1.400000  analytic=1.400000  numeric=1.400000
3 / x                  value=4.285714  AD=-6.122449  analytic=-6.122449  numeric=-6.122449
x ** 3                 value=0.343000  AD=1.470000  analytic=1.470000  numeric=1.470000
2 ** x                 value=1.624505  AD=1.126021  analytic=1.126021  numeric=1.126021
x ** x                 value=0.779056  AD=0.501186  analytic=0.501186  numeric=0.501186
sin(x)                 value=0.644218  AD=0.764842  analytic=0.764842  numeric=0.764842
x*x / (x+1)            value=0.288235  AD=0.653979  analytic=0.653979  numeric=0.653979
cos(x)                 value=0.764842  AD=0.644218  analytic=-0.644218  numeric=-0.644218
tan(x)                 